### Ingesting results .csv file

In [0]:
%run ../00-common/01-environment-config

In [0]:
%run ../00-common/02-bronze_helper

In [0]:
source_file = f"{landing_forlder_path}/results"
table_name = f"{catalog_name}.{bronze_schema}.results"


### Adding Ingestion data

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

results_schema = StructType([
    StructField("date", DateType(), True),
    StructField("raceName", StringType(), True),
    StructField("round", IntegerType(), True),
    StructField("season", IntegerType(), True),
    StructField("url", StringType(), True),
    StructField("constructorId", StringType(), True),
    StructField("driverId", StringType(), True),
    StructField("grid", IntegerType(), True),
    StructField("laps", IntegerType(), True),
    StructField("number", IntegerType(), True),
    StructField("points", DoubleType(), True),
    StructField("position", IntegerType(), True),
    StructField("positionText", StringType(), True),
    StructField("status", StringType(), True)
])

results_df = (
    spark.read.format('json')
    .schema(results_schema)
    .option('mode', 'FAILFAST')
    .load(source_file)
    .select("*", "_metadata")
)

In [0]:
display(results_df)

In [0]:
results_final_df = add_ingestion_metadata(results_df)

### Creating Delta table

In [0]:
(results_final_df
 .write
 .mode("overwrite")
 .format('delta')
 .saveAsTable(table_name)
)
display(spark.read.table(table_name))

In [0]:
%sql
SELECT season, COUNT(*) FROM formula1.bronze.results GROUP BY season ORDER BY season DESC;